# Moser + Dijkstra для кривой границы $y=x^2$

Здесь **только 2D**. Две оптические среды разделены параболой

$$y=x^2.$$

- выше параболы: `N_ABOVE`
- ниже параболы: `N_BELOW`
- Dijkstra минимизирует оптическую длину $\int n\,ds$
- рёбра графа точно разрезаются в пересечениях с параболой
- отдельно считается непрерывное решение Ферма/Снелла

Меняй в основном только первую ячейку.


In [ ]:
#@title 1. Настройки — меняй эту ячейку
from pathlib import Path

# ============================================================
# ТОЧКИ A И B
# Они должны быть по разные стороны от y=x^2
# ============================================================

A_X = -3.0 #@param {type:"number"}
A_Y = 10.0 #@param {type:"number"}

B_X = 3.0  #@param {type:"number"}
B_Y = -1.5 #@param {type:"number"}

# ============================================================
# ПОКАЗАТЕЛИ ПРЕЛОМЛЕНИЯ
#
# ABOVE: y > x^2
# BELOW: y < x^2
# ============================================================

N_ABOVE = 1.0 #@param {type:"number", min:0.01}
N_BELOW = 1.5 #@param {type:"number", min:0.01}

# ============================================================
# ОБЛАСТЬ РАСЧЁТА / ГРАФИКА
# Границы автоматически округляются до кратных H
# ============================================================

X_MIN = -4.0 #@param {type:"number"}
X_MAX = 4.0  #@param {type:"number"}
Y_MIN = -3.0 #@param {type:"number"}
Y_MAX = 12.0 #@param {type:"number"}

# ============================================================
# MOSER + DIJKSTRA
#
# меньше H -> мельче основные клетки
# больше Q -> больше направлений внутри клетки
# ============================================================

H = 1.0 #@param {type:"number", min:0.1}
Q = 7   #@param {type:"integer", min:0}

# Дополнительные точки прямо на y=x^2
INTERFACE_SAMPLES = 260 #@param {type:"integer", min:20}

# Точность независимого непрерывного решения
CONTINUOUS_SCAN_POINTS = 3001 #@param {type:"integer", min:501}

# ============================================================
# ВИЗУАЛИЗАЦИЯ
# ============================================================

SHOW_INTERFACE_NODES = True #@param {type:"boolean"}
SHOW_ALL_NODES = False      #@param {type:"boolean"}

NORMAL_DRAW_LENGTH = 1.2 #@param {type:"number", min:0.1}

SAVE_FIGURE = False #@param {type:"boolean"}
OUTPUT_DIRECTORY = Path("/content/parabola_moser_results")

MAX_MAIN_CELLS = 5000
MAX_GRAPH_NODES = 300000

print("Interface: y = x^2")
print("A =", (A_X, A_Y))
print("B =", (B_X, B_Y))
print("n_above =", N_ABOVE)
print("n_below =", N_BELOW)
print("H =", H, "| Q =", Q)


In [ ]:
#@title 2. Solver — эту ячейку обычно не меняй
"""2-D refraction across a curved interface y = x^2.

Google-Colab friendly implementation of an interface-conforming
Moser + Dijkstra optical ray solver.

The plane is split into two media:

    ABOVE: y > x^2     refractive index N_ABOVE
    BELOW: y < x^2     refractive index N_BELOW

A and B must be on opposite sides of the parabola for the
single-refraction continuous benchmark.

Dijkstra minimizes optical path length:
    OPL = integral n ds

The graph:
1. creates Moser-type boundary nodes in every square cell,
2. inserts exact/sampled nodes on y=x^2,
3. splits every graph chord exactly at intersections with y=x^2,
4. assigns n_above or n_below to every resulting edge piece,
5. runs Dijkstra from A to B.

A continuous Fermat/Snell solution is calculated independently
for comparison.
"""

from __future__ import annotations

from dataclasses import dataclass
import heapq
import math
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np


Point = tuple[float, float]
Medium = Literal["above", "below", "interface"]
EPS = 1e-11


# =====================================================================
# BASIC 2-D GEOMETRY
# =====================================================================

def distance(a: Point, b: Point) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])


def dot(a: Point, b: Point) -> float:
    return a[0] * b[0] + a[1] * b[1]


def subtract(a: Point, b: Point) -> Point:
    return a[0] - b[0], a[1] - b[1]


def normalize(v: Point) -> Point:
    length = math.hypot(v[0], v[1])
    if length <= 1e-14:
        return 0.0, 0.0
    return v[0] / length, v[1] / length


def segment_point(a: Point, b: Point, t: float) -> Point:
    return (
        a[0] + t * (b[0] - a[0]),
        a[1] + t * (b[1] - a[1]),
    )


# =====================================================================
# PARABOLIC INTERFACE y = x^2
# =====================================================================

def interface_y(x: float) -> float:
    return x * x


def interface_measure(p: Point) -> float:
    """Positive above y=x^2, negative below it."""
    return p[1] - p[0] * p[0]


def classify_point(p: Point, tol: float = 2e-9) -> Medium:
    value = interface_measure(p)
    if value > tol:
        return "above"
    if value < -tol:
        return "below"
    return "interface"


def medium_index(medium: Medium) -> float:
    if medium == "above":
        return float(N_ABOVE)
    if medium == "below":
        return float(N_BELOW)
    # A zero-length ideal interface has no independent physical thickness.
    return min(float(N_ABOVE), float(N_BELOW))


def parabola_segment_intersections(a: Point, b: Point) -> tuple[float, ...]:
    """All t in [0,1] where segment a+t(b-a) intersects y=x^2."""
    x0, y0 = a
    dx = b[0] - a[0]
    dy = b[1] - a[1]

    # y0 + dy*t = (x0 + dx*t)^2
    # dx^2 t^2 + (2*x0*dx - dy)t + (x0^2-y0) = 0
    qa = dx * dx
    qb = 2.0 * x0 * dx - dy
    qc = x0 * x0 - y0

    roots: list[float] = []

    if abs(qa) <= 1e-15:
        if abs(qb) > 1e-15:
            roots.append(-qc / qb)
    else:
        disc = qb * qb - 4.0 * qa * qc
        if disc >= -1e-13:
            disc = max(0.0, disc)
            sqrt_disc = math.sqrt(disc)
            roots.append((-qb - sqrt_disc) / (2.0 * qa))
            roots.append((-qb + sqrt_disc) / (2.0 * qa))

    valid = []
    for t in roots:
        if -EPS <= t <= 1.0 + EPS:
            valid.append(min(1.0, max(0.0, t)))

    # de-duplicate tangencies / numerical repeats
    return tuple(sorted(set(round(t, 14) for t in valid)))


def chord_split_parameters(a: Point, b: Point) -> tuple[float, ...]:
    params = [0.0, 1.0]
    for t in parabola_segment_intersections(a, b):
        if EPS < t < 1.0 - EPS:
            params.append(t)
    return tuple(sorted(set(params)))


def classify_edge_piece(a: Point, b: Point) -> Medium:
    return classify_point(segment_point(a, b, 0.5))


def tangent_unit(x: float) -> Point:
    # y'=2x -> tangent (1, 2x)
    return normalize((1.0, 2.0 * x))


def normal_unit(x: float) -> Point:
    # grad(y-x^2)=(-2x,1)
    return normalize((-2.0 * x, 1.0))


# =====================================================================
# PROBLEM / GRAPH DATA
# =====================================================================

@dataclass(frozen=True)
class Problem:
    source: Point
    target: Point
    domain: tuple[float, float, float, float]


@dataclass(frozen=True)
class Edge:
    target: int
    length: float
    optical_length: float
    medium: Medium


@dataclass(frozen=True)
class Graph:
    points: tuple[Point, ...]
    adjacency: tuple[tuple[Edge, ...], ...]
    interface_nodes: tuple[int, ...]
    edge_count: int
    split_chords: int


@dataclass(frozen=True)
class GraphInfo:
    cells_x: int
    cells_y: int
    h: float
    q: int
    lattice_step: float
    nodes: int
    edges: int
    interface_nodes: int
    split_chords: int


def align_domain(
    xmin: float,
    xmax: float,
    ymin: float,
    ymax: float,
    h: float,
) -> tuple[float, float, float, float]:
    return (
        h * math.floor(xmin / h),
        h * math.ceil(xmax / h),
        h * math.floor(ymin / h),
        h * math.ceil(ymax / h),
    )


def configured_problem() -> Problem:
    domain = align_domain(X_MIN, X_MAX, Y_MIN, Y_MAX, H)
    problem = Problem(
        source=(float(A_X), float(A_Y)),
        target=(float(B_X), float(B_Y)),
        domain=domain,
    )

    xmin, xmax, ymin, ymax = domain
    for name, p in (("A", problem.source), ("B", problem.target)):
        if not (xmin <= p[0] <= xmax and ymin <= p[1] <= ymax):
            raise ValueError(f"{name}={p} is outside the plotting/graph domain {domain}.")

    source_medium = classify_point(problem.source)
    target_medium = classify_point(problem.target)

    if source_medium == "interface" or target_medium == "interface":
        raise ValueError("A and B should not lie exactly on y=x^2.")

    if source_medium == target_medium:
        raise ValueError(
            "For this benchmark put A and B on opposite sides of y=x^2. "
            f"Currently both are {source_medium!r}."
        )

    if N_ABOVE <= 0 or N_BELOW <= 0:
        raise ValueError("Refractive indices must be positive.")

    return problem


# =====================================================================
# INTERFACE NODES
# =====================================================================

def cells_containing_coordinate(
    coordinate: float,
    lower: float,
    h: float,
    count: int,
    tol: float = 1e-11,
) -> tuple[int, ...]:
    scaled = (coordinate - lower) / h
    nearest = round(scaled)

    if math.isclose(scaled, nearest, rel_tol=0.0, abs_tol=tol):
        candidates = (nearest - 1, nearest)
    else:
        candidates = (math.floor(scaled),)

    return tuple(i for i in candidates if 0 <= i < count)


def exact_interface_grid_intersections(
    domain: tuple[float, float, float, float],
    h: float,
) -> tuple[Point, ...]:
    """Intersections of y=x^2 with the square-cell grid lines."""
    xmin, xmax, ymin, ymax = domain
    points: list[Point] = []

    # Vertical grid lines x = const.
    nx = round((xmax - xmin) / h)
    for i in range(nx + 1):
        x = xmin + i * h
        y = x * x
        if ymin - EPS <= y <= ymax + EPS:
            points.append((x, y))

    # Horizontal grid lines y = const.
    ny = round((ymax - ymin) / h)
    for j in range(ny + 1):
        y = ymin + j * h
        if y < -EPS:
            continue
        root = math.sqrt(max(y, 0.0))
        for x in (-root, root):
            if xmin - EPS <= x <= xmax + EPS:
                points.append((x, y))

    # de-duplicate
    unique = {
        (round(float(x), 12), round(float(y), 12))
        for x, y in points
    }
    return tuple((x, y) for x, y in sorted(unique))


def sampled_interface_points(
    domain: tuple[float, float, float, float],
    count: int,
) -> tuple[Point, ...]:
    xmin, xmax, ymin, ymax = domain

    if ymax < 0.0:
        return tuple()

    visible_root = math.sqrt(max(0.0, ymax))
    left = max(xmin, -visible_root)
    right = min(xmax, visible_root)

    if right <= left:
        return tuple()

    xs = np.linspace(left, right, max(2, int(count)))
    pts = []
    for x in xs:
        y = x * x
        if ymin - EPS <= y <= ymax + EPS:
            pts.append((float(x), float(y)))
    return tuple(pts)


# =====================================================================
# MOSER GRAPH
# =====================================================================

def build_graph(problem: Problem) -> tuple[Graph, GraphInfo, dict[Point, int]]:
    xmin, xmax, ymin, ymax = problem.domain
    h = float(H)
    q = int(Q)

    if h <= 0.0:
        raise ValueError("H must be positive.")
    if q < 0:
        raise ValueError("Q must be non-negative.")

    cells_x_float = (xmax - xmin) / h
    cells_y_float = (ymax - ymin) / h
    cells_x = round(cells_x_float)
    cells_y = round(cells_y_float)

    if not math.isclose(cells_x_float, cells_x, abs_tol=1e-9):
        raise ValueError("Domain width must be an integer multiple of H.")
    if not math.isclose(cells_y_float, cells_y, abs_tol=1e-9):
        raise ValueError("Domain height must be an integer multiple of H.")

    if cells_x * cells_y > MAX_MAIN_CELLS:
        raise ValueError(
            "Graph domain is too large. Increase H or reduce the domain."
        )

    intervals = q + 1
    lattice_step = h / intervals

    points: list[Point] = []
    adjacency: list[dict[int, Edge]] = []
    coordinate_to_id: dict[Point, int] = {}
    cell_nodes: dict[tuple[int, int], set[int]] = {
        (i, j): set()
        for i in range(cells_x)
        for j in range(cells_y)
    }
    interface_node_ids: set[int] = set()

    def key(p: Point) -> Point:
        return round(float(p[0]), 12), round(float(p[1]), 12)

    def add_node(p: Point, interface: bool = False) -> int:
        k = key(p)
        if k in coordinate_to_id:
            node = coordinate_to_id[k]
            if interface:
                interface_node_ids.add(node)
            return node

        node = len(points)
        coordinate_to_id[k] = node
        points.append((float(p[0]), float(p[1])))
        adjacency.append({})

        if interface:
            interface_node_ids.add(node)

        if len(points) > MAX_GRAPH_NODES:
            raise ValueError(
                "Graph node limit exceeded. Increase H, reduce Q, "
                "or reduce INTERFACE_SAMPLES."
            )

        return node

    def attach_to_cells(node: int, p: Point) -> None:
        for i in cells_containing_coordinate(p[0], xmin, h, cells_x):
            for j in cells_containing_coordinate(p[1], ymin, h, cells_y):
                cell_nodes[(i, j)].add(node)

    # -------------------------------------------------------------
    # 1) Standard Moser boundary nodes on each square cell.
    # -------------------------------------------------------------
    for cell_x in range(cells_x):
        for cell_y in range(cells_y):
            left = cell_x * intervals
            bottom = cell_y * intervals

            integer_keys = set()
            for offset in range(intervals + 1):
                integer_keys.add((left + offset, bottom))
                integer_keys.add((left + offset, bottom + intervals))
                integer_keys.add((left, bottom + offset))
                integer_keys.add((left + intervals, bottom + offset))

            for ix, iy in integer_keys:
                p = (
                    xmin + ix * lattice_step,
                    ymin + iy * lattice_step,
                )
                node = add_node(p)
                cell_nodes[(cell_x, cell_y)].add(node)

    # -------------------------------------------------------------
    # 2) Exact A and B.
    # -------------------------------------------------------------
    for p in (problem.source, problem.target):
        node = add_node(p)
        attach_to_cells(node, p)

    # -------------------------------------------------------------
    # 3) Explicit interface nodes y=x^2.
    #    Exact grid intersections + additional samples.
    # -------------------------------------------------------------
    interface_points = list(
        exact_interface_grid_intersections(problem.domain, h)
    )
    interface_points.extend(
        sampled_interface_points(problem.domain, int(INTERFACE_SAMPLES))
    )

    for p in interface_points:
        node = add_node(p, interface=True)
        attach_to_cells(node, p)

    split_chords = 0

    def add_edge(a_id: int, b_id: int, medium: Medium) -> None:
        if a_id == b_id:
            return

        a = points[a_id]
        b = points[b_id]
        length = distance(a, b)
        if length <= 1e-14:
            return

        opl = medium_index(medium) * length

        existing = adjacency[a_id].get(b_id)
        if existing is not None and existing.optical_length <= opl:
            return

        edge_ab = Edge(b_id, length, opl, medium)
        edge_ba = Edge(a_id, length, opl, medium)
        adjacency[a_id][b_id] = edge_ab
        adjacency[b_id][a_id] = edge_ba

    # -------------------------------------------------------------
    # 4) Complete graph inside each main cell.
    #    Every chord is split exactly at y=x^2.
    # -------------------------------------------------------------
    for local in cell_nodes.values():
        ordered = sorted(local)

        for a_pos, a_id in enumerate(ordered):
            for b_id in ordered[a_pos + 1:]:
                a = points[a_id]
                b = points[b_id]

                params = chord_split_parameters(a, b)
                if len(params) > 2:
                    split_chords += 1

                piece_ids = []
                for t in params:
                    p = segment_point(a, b, t)
                    is_interface = abs(interface_measure(p)) <= 2e-8
                    piece_ids.append(add_node(p, interface=is_interface))

                for first, second in zip(piece_ids, piece_ids[1:]):
                    medium = classify_edge_piece(points[first], points[second])
                    add_edge(first, second, medium)

    frozen_adj = tuple(
        tuple(sorted(neighbors.values(), key=lambda e: e.target))
        for neighbors in adjacency
    )

    graph = Graph(
        points=tuple(points),
        adjacency=frozen_adj,
        interface_nodes=tuple(sorted(interface_node_ids)),
        edge_count=sum(len(n) for n in frozen_adj) // 2,
        split_chords=split_chords,
    )

    info = GraphInfo(
        cells_x=cells_x,
        cells_y=cells_y,
        h=h,
        q=q,
        lattice_step=lattice_step,
        nodes=len(graph.points),
        edges=graph.edge_count,
        interface_nodes=len(graph.interface_nodes),
        split_chords=split_chords,
    )

    point_ids = {
        key(p): node
        for node, p in enumerate(graph.points)
    }

    return graph, info, point_ids


# =====================================================================
# DIJKSTRA
# =====================================================================

@dataclass(frozen=True)
class DistanceField:
    distances: tuple[float, ...]
    predecessors: tuple[int | None, ...]

    def path_to(self, target: int) -> tuple[int, ...]:
        if math.isinf(self.distances[target]):
            raise ValueError("Target is unreachable.")

        path = [target]
        while self.predecessors[path[-1]] is not None:
            path.append(self.predecessors[path[-1]])
        path.reverse()
        return tuple(path)


def find_node(point_ids: dict[Point, int], p: Point) -> int:
    key = round(p[0], 12), round(p[1], 12)
    return point_ids[key]


def dijkstra(
    graph: Graph,
    source: int,
    target: int,
) -> DistanceField:
    n = len(graph.points)
    dist = [math.inf] * n
    prev: list[int | None] = [None] * n
    settled = [False] * n

    dist[source] = 0.0
    heap = [(0.0, source)]

    while heap:
        current, node = heapq.heappop(heap)

        if settled[node]:
            continue

        settled[node] = True

        if node == target:
            break

        for edge in graph.adjacency[node]:
            candidate = current + edge.optical_length

            if candidate < dist[edge.target]:
                dist[edge.target] = candidate
                prev[edge.target] = node
                heapq.heappush(heap, (candidate, edge.target))

    return DistanceField(tuple(dist), tuple(prev))


def edge_between(graph: Graph, a: int, b: int) -> Edge:
    for edge in graph.adjacency[a]:
        if edge.target == b:
            return edge
    raise KeyError((a, b))


def solve_graph(
    graph: Graph,
    point_ids: dict[Point, int],
    problem: Problem,
) -> dict[str, object]:
    source_id = find_node(point_ids, problem.source)
    target_id = find_node(point_ids, problem.target)

    field = dijkstra(graph, source_id, target_id)
    path = field.path_to(target_id)

    path_points = tuple(graph.points[node] for node in path)
    path_edges = tuple(
        edge_between(graph, a, b)
        for a, b in zip(path, path[1:])
    )

    opl = sum(edge.optical_length for edge in path_edges)

    # Count true above<->below transitions, ignoring rare interface labels.
    media = [edge.medium for edge in path_edges if edge.medium != "interface"]
    crossings = sum(
        1 for m1, m2 in zip(media, media[1:])
        if m1 != m2
    )

    return {
        "path": path,
        "path_points": path_points,
        "path_edges": path_edges,
        "optical_path": opl,
        "crossings": crossings,
    }


# =====================================================================
# EXACT SINGLE-REFRACTION FERMAT / SNELL BENCHMARK
# =====================================================================

def segment_measure_extrema(a: Point, b: Point) -> tuple[float, float]:
    """Exact min/max of y(t)-x(t)^2 along a straight segment."""
    x0, y0 = a
    dx = b[0] - a[0]
    dy = b[1] - a[1]

    qa = -(dx * dx)
    qb = dy - 2.0 * x0 * dx
    qc = y0 - x0 * x0

    values = [qc, qa + qb + qc]

    if abs(qa) > 1e-15:
        t_vertex = -qb / (2.0 * qa)
        if 0.0 < t_vertex < 1.0:
            values.append(
                qa * t_vertex * t_vertex
                + qb * t_vertex
                + qc
            )

    return min(values), max(values)


def segment_stays_in_medium(
    a: Point,
    b: Point,
    medium: Medium,
    tol: float = 2e-9,
) -> bool:
    minimum, maximum = segment_measure_extrema(a, b)

    if medium == "above":
        return minimum >= -tol
    if medium == "below":
        return maximum <= tol
    return False


def continuous_opl_at_x(
    x: float,
    problem: Problem,
) -> float:
    p = (x, x * x)

    source_medium = classify_point(problem.source)
    target_medium = classify_point(problem.target)

    if not segment_stays_in_medium(problem.source, p, source_medium):
        return math.inf

    if not segment_stays_in_medium(p, problem.target, target_medium):
        return math.inf

    return (
        medium_index(source_medium) * distance(problem.source, p)
        + medium_index(target_medium) * distance(p, problem.target)
    )


def golden_section_minimize(
    f,
    left: float,
    right: float,
    iterations: int = 90,
) -> tuple[float, float]:
    golden = (math.sqrt(5.0) - 1.0) / 2.0

    x1 = right - golden * (right - left)
    x2 = left + golden * (right - left)
    f1 = f(x1)
    f2 = f(x2)

    for _ in range(iterations):
        if f1 <= f2:
            right = x2
            x2, f2 = x1, f1
            x1 = right - golden * (right - left)
            f1 = f(x1)
        else:
            left = x1
            x1, f1 = x2, f2
            x2 = left + golden * (right - left)
            f2 = f(x2)

    x = 0.5 * (left + right)
    return x, f(x)


@dataclass(frozen=True)
class ContinuousReference:
    crossing: Point
    optical_path: float
    snell_residual: float
    path: tuple[Point, Point, Point]


def continuous_reference(problem: Problem) -> ContinuousReference:
    xmin, xmax, ymin, ymax = problem.domain

    if ymax < 0:
        raise ValueError("The visible domain never reaches y=x^2.")

    visible_root = math.sqrt(max(ymax, 0.0))
    left = max(xmin, -visible_root)
    right = min(xmax, visible_root)

    xs = np.linspace(left, right, int(CONTINUOUS_SCAN_POINTS))
    costs = np.array(
        [continuous_opl_at_x(float(x), problem) for x in xs],
        dtype=float,
    )

    finite = np.isfinite(costs)
    if not finite.any():
        raise ValueError(
            "No valid one-crossing continuous path found inside the domain. "
            "Expand X/Y bounds or move A/B."
        )

    best = int(np.nanargmin(costs))

    lo_i = max(0, best - 2)
    hi_i = min(len(xs) - 1, best + 2)

    local_left = float(xs[lo_i])
    local_right = float(xs[hi_i])

    x_star, opl = golden_section_minimize(
        lambda x: continuous_opl_at_x(x, problem),
        local_left,
        local_right,
    )

    p = (x_star, x_star * x_star)

    source_medium = classify_point(problem.source)
    target_medium = classify_point(problem.target)

    incoming = normalize(subtract(p, problem.source))
    outgoing = normalize(subtract(problem.target, p))
    tangent = tangent_unit(x_star)

    # Fermat/Snell tangential momentum continuity:
    # n1 (d_in · t) = n2 (d_out · t)
    residual = abs(
        medium_index(source_medium) * dot(incoming, tangent)
        - medium_index(target_medium) * dot(outgoing, tangent)
    )

    return ContinuousReference(
        crossing=p,
        optical_path=opl,
        snell_residual=residual,
        path=(problem.source, p, problem.target),
    )


# =====================================================================
# PLOTTING
# =====================================================================

def plot_solution(
    graph: Graph,
    problem: Problem,
    result: dict[str, object],
    reference: ContinuousReference,
) -> plt.Figure:
    xmin, xmax, ymin, ymax = problem.domain

    fig, ax = plt.subplots(figsize=(10, 8))

    xs = np.linspace(xmin, xmax, 1200)
    ys = xs * xs

    # Shade the two media.
    ax.fill_between(
        xs,
        ys,
        ymax,
        where=(ys <= ymax),
        alpha=0.08,
        label=f"above: n={N_ABOVE:g}",
    )
    ax.fill_between(
        xs,
        ymin,
        ys,
        where=(ys >= ymin),
        alpha=0.08,
        label=f"below: n={N_BELOW:g}",
    )

    # Parabolic interface.
    visible = (ys >= ymin) & (ys <= ymax)
    ax.plot(
        xs[visible],
        ys[visible],
        linewidth=1.8,
        label=r"interface $y=x^2$",
    )

    # Inserted / split interface nodes.
    interface_points = [
        graph.points[node]
        for node in graph.interface_nodes
        if xmin <= graph.points[node][0] <= xmax
        and ymin <= graph.points[node][1] <= ymax
    ]

    if SHOW_INTERFACE_NODES and interface_points:
        ax.scatter(
            [p[0] for p in interface_points],
            [p[1] for p in interface_points],
            s=8,
            alpha=0.28,
            label="inserted interface nodes",
        )

    # Optional all graph nodes.
    if SHOW_ALL_NODES:
        ax.scatter(
            [p[0] for p in graph.points],
            [p[1] for p in graph.points],
            s=2,
            alpha=0.10,
            linewidths=0,
            label="all graph nodes",
        )

    # Continuous Fermat/Snell ray.
    cp = reference.path
    ax.plot(
        [p[0] for p in cp],
        [p[1] for p in cp],
        "--",
        linewidth=1.8,
        label="continuous Snell/Fermat ray",
    )

    # Moser-Dijkstra ray.
    gp = result["path_points"]
    ax.plot(
        [p[0] for p in gp],
        [p[1] for p in gp],
        "o-",
        markersize=3,
        linewidth=2.0,
        label="conforming Moser-Dijkstra ray",
    )

    # A and B.
    ax.scatter(
        [problem.source[0], problem.target[0]],
        [problem.source[1], problem.target[1]],
        s=60,
        facecolors="white",
        edgecolors="black",
        zorder=10,
    )
    ax.annotate(
        "A",
        problem.source,
        xytext=(-13, 8),
        textcoords="offset points",
        fontsize=12,
    )
    ax.annotate(
        "B",
        problem.target,
        xytext=(8, -15),
        textcoords="offset points",
        fontsize=12,
    )

    # P and the local normal.
    p = reference.crossing
    ax.scatter([p[0]], [p[1]], s=35, zorder=11)
    ax.annotate(
        "P",
        p,
        xytext=(-18, 10),
        textcoords="offset points",
        fontsize=12,
    )

    n = normal_unit(p[0])
    normal_half_length = float(NORMAL_DRAW_LENGTH)
    n1 = (
        p[0] - normal_half_length * n[0],
        p[1] - normal_half_length * n[1],
    )
    n2 = (
        p[0] + normal_half_length * n[0],
        p[1] + normal_half_length * n[1],
    )

    ax.plot(
        [n1[0], n2[0]],
        [n1[1], n2[1]],
        ":",
        linewidth=1.3,
        label="surface normal",
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(
        r"Curved-interface benchmark: $y=x^2$"
        + f"\nH={H:g}, Q={Q}, interface samples={INTERFACE_SAMPLES}"
    )
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, fontsize=8, loc="best")
    fig.tight_layout()

    return fig


# =====================================================================
# DRIVER
# =====================================================================

def main() -> dict[str, object]:
    problem = configured_problem()

    graph, info, point_ids = build_graph(problem)
    result = solve_graph(graph, point_ids, problem)
    reference = continuous_reference(problem)

    relative_error = abs(
        result["optical_path"] - reference.optical_path
    ) / max(reference.optical_path, 1e-12)

    print("=== PARABOLIC INTERFACE y = x^2 ===")
    print(f"A = {problem.source}   medium = {classify_point(problem.source)}")
    print(f"B = {problem.target}   medium = {classify_point(problem.target)}")
    print(f"n_above = {N_ABOVE:g}   n_below = {N_BELOW:g}")

    print()
    print("=== GRAPH ===")
    print(f"cells = {info.cells_x} x {info.cells_y}")
    print(f"H={info.h:g}   Q={info.q}   lattice step={info.lattice_step:.6f}")
    print(f"nodes={info.nodes}")
    print(f"edges={info.edges}")
    print(f"interface nodes={info.interface_nodes}")
    print(f"split chords={info.split_chords}")

    print()
    print("=== MOSER-DIJKSTRA ===")
    print(f"optical path = {result['optical_path']:.9f}")
    print(f"medium transitions = {result['crossings']}")

    print()
    print("=== CONTINUOUS FERMAT/SNELL ===")
    print(f"P = ({reference.crossing[0]:.9f}, {reference.crossing[1]:.9f})")
    print(f"optical path = {reference.optical_path:.9f}")
    print(f"Snell tangential residual = {reference.snell_residual:.3e}")

    print()
    print("=== COMPARISON ===")
    print(f"relative OPL error = {relative_error:.6e}")

    tolerance = 0.02 + 4.0 / (Q + 1) ** 2
    status = "PASS" if relative_error <= tolerance else "REFINE H/Q"
    print(f"self-check = {status}   tolerance={tolerance:.6e}")

    fig = plot_solution(graph, problem, result, reference)

    if SAVE_FIGURE:
        OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
        output = OUTPUT_DIRECTORY / "parabola_y_eq_x2_moser_dijkstra.png"
        fig.savefig(output, dpi=250, bbox_inches="tight")
        print(f"saved: {output}")

    plt.show()

    return {
        "problem": problem,
        "graph": graph,
        "graph_info": info,
        "result": result,
        "reference": reference,
        "relative_error": relative_error,
    }


In [ ]:
#@title 3. Запустить
solution = main()
